In [ ]:
import ROOT
import numpy as np
import pandas as pd

#funzione modello
def model(p, A, B, C):
    return B + (C(p-A))/((p-A)-C)

def model_derivative(p, A, B, C):
    dx=10e-6
    return (model(p + dx, A, B, C) - model(p, A, B, C)) / dx

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy()

#definizione variabili
p = df[:, 0]
q = df[:, 1]
err_p = df[:, 2]
err_q = df[:,3]

#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0, 0)
#model_fit.SetParameter(2, 0.0)
model_fit.SetParLimits(0, 0, None) 

model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati con errori; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerColor(ROOT.kBlue)
gr1.SetLineColor(ROOT.kBlack)

fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = gr1.GetParameters()
errors = gr1.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + (model_derivative(p, *model_fit.Parameters) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {parameters[0]} ± {errors[0]}")
print(f"B = {parameters[1]} ± {errors[1]}")
print(f"C = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("APX")

c1.Draw()
c1.SaveAs("fit_root.png")

In [ ]:
err_R = 1
err_L = 1
err_x0 = 0,115470053837925

if parameters[0] < 3*(errors[0] + err_x0 + err_R + err_L):
    print("A è compatibile con 0")
else :
    print("A NON è compatibile con 0")

print("---")

if parameters[1] < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


In [ ]:
residuals_norm = (q - model(p, *fit_res.Parameters())) / err_qeff

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kGreen+2)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kBlue)
hist.SetFillColor(ROOT.kBlue)
hist.GetXaxis().SetTitle("valore")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "SR")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = hist.GetParameters()
errors = hist.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

c2.Draw()
c2.SaveAs("hist.png")

In [ ]:
if parameters[0] - 1 < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if parameters[1] < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if parameters[2] - 1 < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

In [ ]:
def line(x, m, q):
    return m * x + q

residuals = q - model(p, *fit_res.Parameters())

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Dati con errori; x; y")
gr2.SetMarkerStyle(20)
gr2.SetMarkerColor(ROOT.kBlue)
gr2.SetLineColor(ROOT.kBlack)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = gr2.GetParameters()
errors = gr2.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")


c3.Draw()
c3.SaveAs("fit_line.png")